The Auditing Engine is running and the search functions are fast. But Sanaa calls you into her office with a whiteboard already covered in red marker.

"Here's the problem," she says, drawing a box labeled `dict`. "Every field record is a dictionary. Nothing stops anyone from setting `Pollution_level` to 7,000. Nothing ensures `Plot_size` is always there. Nothing groups the behavior that belongs with the data." She draws a line through the box. "A dictionary is fine for one-off scripts. We are building a system that five other engineers will build on top of. Loose data breaks systems."

She turns to you. "We need to redesign."

This is where **object-oriented programming** comes in. Instead of passing dictionaries around and hoping everyone handles them consistently, we define a class — a blueprint that declares what every field *has* (attributes) and what every field *can do* (methods). The blueprint enforces structure. Other code can trust it.

By the end of this session you will have built the **Maji Ndogo Field Registry**: a set of Python classes that model the survey data with proper validation, inheritance, and polymorphism — the kind of code that professional engineering teams actually build on.
    """)

In [1]:
def download():
    
    from urllib.request import urlretrieve
    import os

    url = (
        "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/"
        "Maji_Ndogo/Maji_Ndogo_farm_survey_small.db"
    )

    db_file = "Maji_Ndogo_farm_survey_small.db"

    # Download only if the database is not already present
    if not os.path.exists(db_file):
        urlretrieve(url, db_file)
        print(f"Downloaded '{db_file}'.")
    else:
        print(f"'{db_file}' already exists.")

    return

In [2]:
def data_base():
    ### DO NOT CHANGE ANYTHING IN THIS CELL, ONLY EXECUTE IT!

    import sqlite3

    connection = sqlite3.connect('Maji_Ndogo_farm_survey_small.db')
    cursor = connection.cursor()
    rows = cursor.execute(
        """SELECT Field_ID, Pollution_level, Plot_size, Crop_type, Annual_yield, Standard_yield
           FROM farm_management_features"""
    ).fetchall()
    connection.close()

    field_data = []
    for field_id, pollution, plot_size, yield_value, crop_name, standard in rows:
        # Note: the source columns 'Crop_type' and 'Annual_yield' were swapped
        # during export, so we swap them back as we build each record.
        field_data.append({
            'Field_ID': int(field_id),
            'Pollution_level': round(pollution, 2),
            'Plot_size': round(plot_size, 1),
            'Crop_type': crop_name.strip(),
            'Annual_yield': round(yield_value, 2),
            'Standard_yield': round(standard, 2),
        })

    print(f"Loaded {len(field_data)} field survey records.")
    print("First record:", field_data[0])
    return (field_data,)

In [29]:
field_data = data_base()[0]

Loaded 5654 field survey records.
First record: {'Field_ID': 40734, 'Pollution_level': 0.09, 'Plot_size': 1.3, 'Crop_type': 'cassava', 'Annual_yield': 0.75, 'Standard_yield': 0.58}


In [33]:
print(type(field_data))
print(len(field_data))
print(field_data[0])

<class 'list'>
5654
{'Field_ID': 40734, 'Pollution_level': 0.09, 'Plot_size': 1.3, 'Crop_type': 'cassava', 'Annual_yield': 0.75, 'Standard_yield': 0.58}


## __Challenge 1: The `Field` class__

A class is a blueprint. Define it once, and every field in the survey can become an object.

Sensors also malfunction. A reading of `7.5` on a 0–1 scale is physically impossible. **Encapsulation** lets a class control its own data: a `@property` with a setter intercepts every write and enforces the valid range. A field's identity and its self-protection belong to the same blueprint, so you build both here.

### __Task__

Complete the class `Field`:
 - `__init__` must accept `field_id`, `plot_size`, `standard_yield`, and `pollution_level` (**defaulting to `0.0`** — a field with no reading is treated as clean), storing each as an instance attribute with the same PEP 8 `snake_case` name.
- `describe()` must **return** (not print) the string: `'Field {field_id}: {plot_size} Ha (standard yield {standard_yield})'`
- `calculate_yield()` must **return** `round(plot_size * standard_yield, 2)`.
- Back `pollution_level` with a `@property` and a private attribute `_pollution_level`. The setter must raise a `ValueError` with the message `'Pollution level must be between 0 and 1.'` if the value falls outside `[0, 1]`.
- Add a class docstring and a docstring for each method.

> ⚠️ Do not change the class or method names.

 ### __Expected outputs__

> **Input 1:** `f = Field(39924, 3.4, 0.65); f.describe()` → `'Field 39924: 3.4 Ha (standard yield 0.65)'`

> **Input 2:** `Field(39924, 3.4, 0.65).calculate_yield()` → `2.21`

> **Input 3:** `Field(40734, 1.3, 0.58, 0.09).pollution_level` → `0.09`

> **Input 4:**
    ```python
    try:
        Field(40734, 1.3, 0.58, 7.5)
    except ValueError as e:
        print(e)
    ```
    `Pollution level must be between 0 and 1.`

In [3]:
class Field:
    """Represents a surveyed field with validated pollution monitoring."""

    def __init__(self, field_id, plot_size, standard_yield, pollution_level=0.0):
        """Initialize a Field object with its attributes."""
        self.field_id = field_id
        self.plot_size = plot_size
        self.standard_yield = standard_yield
        self.pollution_level = pollution_level  # Uses the property setter

    @property
    def pollution_level(self):
        """Return the pollution level of the field."""
        return self._pollution_level

    @pollution_level.setter
    def pollution_level(self, value):
        """Validate and set the pollution level."""
        if not 0 <= value <= 1:
            raise ValueError("Pollution level must be between 0 and 1.")
        self._pollution_level = value

    def describe(self):
        """Return a description of the field."""
        return (
            f"Field {self.field_id}: "
            f"{self.plot_size} Ha "
            f"(standard yield {self.standard_yield})"
        )

    def calculate_yield(self):
        """Calculate and return the expected yield."""
        return round(self.plot_size * self.standard_yield, 2)

In [4]:
f = Field(39924, 3.4, 0.65)
print(f.describe())

Field 39924: 3.4 Ha (standard yield 0.65)


In [5]:
print(Field(39924, 3.4, 0.65).calculate_yield())

2.21


In [6]:
print(Field(40734, 1.3, 0.58, 0.09).pollution_level)

0.09


In [7]:
try:
    Field(40734, 1.3, 0.58, 7.5) 
except ValueError as e:     
    print(e) 

Pollution level must be between 0 and 1.


## __Challenge 2: Inheritance — crop-specific subclasses__

Tea, coffee, and wheat each sell at a different price per ton. **Inheritance** lets us create specialized subclasses that extend `Field` with crop-specific behavior without rewriting the shared logic. Because each subclass implements the same method names, code can call `describe()` or `projected_revenue()` on any of them without knowing which crop it is — that is **polymorphism**.

### __Task__

Create three subclasses of `Field`: `TeaField` ($1,180/ton), `CoffeeField` ($2,400/ton), `WheatField` ($320/ton). Each must:
- Store the price as a class attribute `PRICE_PER_TON`.
- Override `describe()` to prefix with the crop name (e.g. `'Tea field 39924: ...'`).
- Add a method `projected_revenue()` returning `round(calculate_yield() * PRICE_PER_TON, 2)`.
- Include a class docstring.

> ⚠️ Do not change the class or method names.

### __Expected outputs__

> **Input 1:** `TeaField(39924, 3.4, 0.65, 0.36).describe()` → `'Tea field 39924: 3.4 Ha (standard yield 0.65)'`

> **Input 2:** `TeaField(39924, 3.4, 0.65, 0.36).projected_revenue()` → `2607.8`
    """)

In [8]:
class TeaField(Field):
    """Tea crop field."""
    PRICE_PER_TON = 1180

    def describe(self):
        """Return a description of the tea field."""
        return (
            f"Tea field {self.field_id}: "
            f"{self.plot_size} Ha "
            f"(standard yield {self.standard_yield})"
        )

    def projected_revenue(self):
        """Calculate and return the projected revenue for the tea field."""
        return round(self.calculate_yield() * self.PRICE_PER_TON, 2)


class CoffeeField(Field):
    """Coffee crop field."""
    PRICE_PER_TON = 2400

    def describe(self):
        """Return a description of the coffee field."""
        return (
            f"Coffee field {self.field_id}: "
            f"{self.plot_size} Ha "
            f"(standard yield {self.standard_yield})"
        )

    def projected_revenue(self):
        """Calculate and return the projected revenue for the coffee field."""
        return round(self.calculate_yield() * self.PRICE_PER_TON, 2)


class WheatField(Field):
    """Wheat crop field."""
    PRICE_PER_TON = 320

    def describe(self):
        """Return a description of the wheat field."""
        return (
            f"Wheat field {self.field_id}: "
            f"{self.plot_size} Ha "
            f"(standard yield {self.standard_yield})"
        )

    def projected_revenue(self):
        """Calculate and return the projected revenue for the wheat field."""
        return round(self.calculate_yield() * self.PRICE_PER_TON, 2)

In [9]:
tea = TeaField(39924, 3.4, 0.65, 0.36).describe()
print(tea)

Tea field 39924: 3.4 Ha (standard yield 0.65)


In [10]:
tea1 = TeaField(39924, 3.4, 0.65, 0.36).projected_revenue()
print(tea1)

2607.8


In [11]:
coffee = CoffeeField(40734, 1.3, 0.58)
print(coffee.describe())

Coffee field 40734: 1.3 Ha (standard yield 0.58)


In [12]:
t = TeaField(39924, 3.4, 0.65) 
c = CoffeeField(39924, 3.4, 0.65)
print(c.projected_revenue() - t.projected_revenue())

2696.2


## __Challenge 3: Abstraction — an abstract base class__

Nothing prevents a developer from creating a plain `AbstractCropField` object when they should have used a concrete subclass. **Abstraction** via an abstract base class (ABC) makes that impossible — the class cannot be instantiated until its abstract methods are implemented.

### __Task__

Create `AbstractCropField(ABC)` with:
- `__init__` accepting `field_id`, `plot_size`, and `standard_yield`.
- An abstract method `crop_name()` — every concrete subclass must implement it.
- `calculate_yield()` returning `round(plot_size * standard_yield, 2)`.
- A class docstring.

Then create `MaizeField(AbstractCropField)` implementing `crop_name()` to return `'maize'`.

> ⚠️ Do not change the class or method names.

### __Expected outputs__

> **Input 1:** instantiating the abstract base directly raises a `TypeError` (the exact message wording depends on your Python version):
    ```python
    try:
        AbstractCropField(41964, 4.1, 0.55)
    except TypeError as e:
        print(e)
    ```
    `Can't instantiate abstract class AbstractCropField without an implementation for abstract method 'crop_name'`

> **Input 2:** `MaizeField(41964, 4.1, 0.55).crop_name()` → `'maize'`

> **Input 3:** `MaizeField(41964, 4.1, 0.55).calculate_yield()` → `2.25`

In [13]:
from abc import ABC, abstractmethod


class AbstractCropField(ABC):
    """Abstract base - every concrete crop field must implement crop_name()."""

    def __init__(self, field_id, plot_size, standard_yield):
        """Initialize the crop field."""
        self.field_id = field_id
        self.plot_size = plot_size
        self.standard_yield = standard_yield

    @abstractmethod
    def crop_name(self):
        """Return the name of the crop."""
        pass

    def calculate_yield(self):
        """Calculate and return the expected yield."""
        return round(self.plot_size * self.standard_yield, 2)


class MaizeField(AbstractCropField):
    """Concrete maize field."""

    def crop_name(self):
        """Return the crop name."""
        return "maize"

In [14]:
try:
    AbstractCropField(41964, 4.1, 0.55) 
except TypeError as e:     
    print(e) 

Can't instantiate abstract class AbstractCropField without an implementation for abstract method 'crop_name'


In [15]:
MaizeField(41964, 4.1, 0.55).crop_name()

'maize'

In [16]:
MaizeField(41964, 4.1, 0.55).calculate_yield() 

2.25

In [17]:
TeaField(39924, 3.4, 0.65).calculate_yield()

2.21

## __Challenge 4: The `build_field` factory__

The survey hands you raw dictionaries. Something has to decide which class each record becomes, and that decision belongs in one place rather than scattered across every caller. A **factory function** takes a record and hands back the right object.

### __Task__

Complete `build_field(record)`:
- Accept a raw survey record dictionary.
- Return the correct `TeaField`, `CoffeeField`, or `WheatField` object based on `Crop_type` (case-insensitive).
- Return `None` for any crop not in the pricing catalog.
- Include a docstring.

> ⚠️ Do not change the function name.

### __Expected outputs__

> **Input 1:**
    ```python
    tea_record = {'Field_ID': 39924, 'Plot_size': 3.4, 'Standard_yield': 0.65,
                  'Crop_type': 'tea', 'Pollution_level': 0.36}
    type(build_field(tea_record)).__name__
    ```
    → `'TeaField'`

> **Input 2:**
    ```python
    cassava_record = {'Field_ID': 41964, 'Plot_size': 4.1, 'Standard_yield': 0.55,
                      'Crop_type': 'cassava', 'Pollution_level': 0.2}
    build_field(cassava_record)
    ```
    → `None`

In [23]:
def build_field(record):
    # Insert your code here
    crop_classes = {
        "tea": TeaField,
        "coffee": CoffeeField,
        "wheat": WheatField,
    }
    
    crop = record["Crop_type"].strip().lower()
    
    field_class = crop_classes.get(crop)
    
    if field_class is None:
        return None

    return field_class(
        record["Field_ID"],
        record["Plot_size"],
        record["Standard_yield"],
        record.get("Pollution_level", 0.0)
    )

In [24]:
tea_record = {'Field_ID': 39924, 
              'Plot_size': 3.4, 
              'Standard_yield': 0.65, 
              'Crop_type': 'tea', 
              'Pollution_level': 0.36} 

type(build_field(tea_record)).__name__

'TeaField'

In [25]:
cassava_record = {'Field_ID': 41964, 
                  'Plot_size': 4.1, 
                  'Standard_yield': 0.55,                   
                  'Crop_type': 'cassava', 
                  'Pollution_level': 0.2} 

print(build_field(cassava_record))

None


## __Challenge 5: The `FieldRegistry`__

Now bring it together. The registry aggregates every priced field, calling the same method names on each one regardless of crop. That shared interface is polymorphism doing the real work.

### __Task__

Complete the `FieldRegistry` class:
- Accept a list of raw records in `__init__`, use `build_field()` to build `self.fields` (excluding `None` values).
- `size()` → number of priced fields.
- `total_projected_yield()` → sum of all `calculate_yield()` values, rounded to 2 dp.
- `total_projected_revenue()` → sum of all `projected_revenue()` values, rounded to 2 dp.
- Include a class docstring and method docstrings.

> ⚠️ Do not change the class or method names.

### __Expected outputs (on the full 5,654-record survey)__

> **Input 1:** `registry = FieldRegistry(field_data); registry.size()` → `2840`

> **Input 2:** `registry.total_projected_yield()` → `6103.59`

> **Input 3:** `registry.total_projected_revenue()` → `6412228.0`
    """)

In [26]:
class FieldRegistry:
    """Stores and manages priced crop fields."""

    def __init__(self, records):
        """
        Build field objects from raw records and store only
        fields that have a supported price.

        Parameters
        ----------
        records : list
            List of survey record dictionaries.
        """
        self.fields = []

        for record in records:
            field = build_field(record)

            if field is not None:
                self.fields.append(field)

    def size(self):
        return len(self.fields)

    def total_projected_yield(self):
        return round(
            sum(field.calculate_yield() for field in self.fields),
            2
        )

    def total_projected_revenue(self):
        return round(
            sum(field.projected_revenue() for field in self.fields),
            2
        )

In [30]:
registry = FieldRegistry(field_data)
registry.size()

2840

In [31]:
registry.total_projected_yield()

6103.59

In [32]:
registry.total_projected_revenue()

6412228.0

__DataFrame__

In [34]:
def download():
    
    from urllib.request import urlretrieve
    import os

    url = (
        "https://raw.githubusercontent.com/Explore-AI/Public-Data/master/"
        "Maji_Ndogo/Maji_Ndogo_farm_survey_small.db"
    )

    db_file = "Maji_Ndogo_farm_survey_small.db"

    # Download only if the database is not already present
    if not os.path.exists(db_file):
        urlretrieve(url, db_file)
        print(f"Downloaded '{db_file}'.")
    else:
        print(f"'{db_file}' already exists.")

    return

In [35]:
field_data = data_base()[0]

Loaded 5654 field survey records.
First record: {'Field_ID': 40734, 'Pollution_level': 0.09, 'Plot_size': 1.3, 'Crop_type': 'cassava', 'Annual_yield': 0.75, 'Standard_yield': 0.58}


In [36]:
import pandas as pd

In [37]:
field = pd.DataFrame(field_data)
field.head()

,Field_ID,Pollution_level,Plot_size,Crop_type,Annual_yield,Standard_yield
0,40734,0.09,1.3,cassava,0.75,0.58
1,30629,0.40,2.2,cassava,1.07,0.49
2,39924,0.36,3.4,tea,2.21,0.65
3,5754,0.29,2.4,cassava,1.28,0.53
4,14146,0.04,1.5,wheat,0.83,0.56


In [ ]:
field.to_csv("")